In [ ]:
# INSTALAR LIBRERIAS
!pip install jinja2 weasyprint python-docx -q
!apt-get install -y libharfbuzz-subset0 -qq > /dev/null 2>&1
print("Librerias instaladas correctamente")

Librerias instaladas correctamente


In [ ]:
# CREAR ESTRUCTURA DE CARPETAS
from pathlib import Path

BASE = Path('/content/proyecto_datos_ia')

carpetas = [
    'datos/procesados', 'datos/gobernanza', 'datos/geo',
    'documentos/reporte', 'evidencias/reporte',
    'visualizaciones', 'exportaciones', 'templates', 'scripts',
]
for c in carpetas:
    (BASE / c).mkdir(parents=True, exist_ok=True)

ruta_limpio = BASE / 'datos' / 'procesados'
print("Estructura creada en:", BASE)
print("ruta_limpio   =", ruta_limpio)

Estructura creada en: /content/proyecto_datos_ia
ruta_limpio   = /content/proyecto_datos_ia/datos/procesados


In [ ]:
# SUBIR dataset_limpio.csv  (Guía 4 - gobernanza)
from google.colab import files

print("Sube el archivo: dataset_limpio.csv")
uploaded = files.upload()

for nombre, contenido in uploaded.items():
    (ruta_limpio / nombre).write_bytes(contenido)
    print(f"  {nombre} → datos/procesados/")

Sube el archivo: dataset_limpio.csv


Saving dataset_limpio.csv to dataset_limpio (2).csv
  dataset_limpio (2).csv → datos/procesados/


In [ ]:
# IMPORTAR LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace
from jinja2 import Template
import base64, json, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# CARGAR DATASET PROTEGIDO (Usa ruta_archivo para evitar sobreescribir la carpeta base ruta_limpio)
ruta_archivo = ruta_limpio / "dataset_limpio.csv"

# Si se subió con un nombre duplicado o diferente en Colab, intentamos buscarlo de manera flexible
if not ruta_archivo.exists():
    archivos_csv = list(ruta_limpio.glob("dataset_limpio*.csv"))
    if archivos_csv:
        ruta_archivo = archivos_csv[-1] # Toma el más reciente subido

dataset_limpio = pd.read_csv(ruta_archivo)   # mismo nombre que la tabla usada en las medidas DAX

# VALIDAR QUE EXISTAN LAS COLUMNAS QUE USAN LAS MEDIDAS DAX
columnas_dax = ['tiempo_entrega_min', 'densidad_trafico', 'festival', 'calificacion_repartidor']
faltantes = [c for c in columnas_dax if c not in dataset_limpio.columns]
if faltantes:
    raise KeyError(f"Faltan columnas requeridas por las medidas DAX: {faltantes}\n"
                   f"Columnas disponibles: {list(dataset_limpio.columns)}")

# PREPARAR TIPOS (equivale al tipo de dato que Power BI asigna a cada columna)
dataset_limpio['tiempo_entrega_min']      = pd.to_numeric(dataset_limpio['tiempo_entrega_min'], errors='coerce')
dataset_limpio['calificacion_repartidor'] = pd.to_numeric(dataset_limpio['calificacion_repartidor'], errors='coerce')
for col in ['densidad_trafico', 'festival']:
    dataset_limpio[col] = (dataset_limpio[col].astype(str).str.strip()   # evita "High " ≠ "High"
                           .replace({'nan': 'Sin dato', 'NaN': 'Sin dato', 'None': 'Sin dato', '': 'Sin dato'}))

print(f"Dataset protegido cargado desde: {ruta_archivo.name}")
print(f"Dataset protegido: {dataset_limpio.shape[0]} filas × {dataset_limpio.shape[1]} columnas")
print("\nValores de densidad_trafico:", sorted(dataset_limpio['densidad_trafico'].unique()))
print("Valores de festival:        ", sorted(dataset_limpio['festival'].unique()))
print("\nPrimeras 3 filas:")
print(dataset_limpio.head(3).to_string())

Dataset protegido cargado desde: dataset_limpio.csv
Dataset protegido: 90000 filas × 27 columnas

Valores de densidad_trafico: ['Desconocido', 'High', 'Jam', 'Low', 'Medium']
Valores de festival:         ['No', 'Yes']

Primeras 3 filas:
       id   id_repartidor  edad_repartidor  calificacion_repartidor fecha_pedido hora_pedido hora_recogida_pedido condiciones_climaticas densidad_trafico tipo_pedido festival         ciudad  tiempo_entrega_min  latitud_restaurante  longitud_restaurante  latitud_ubicacion_entrega  longitud_ubicacion_entrega  condicion_vehiculo tipo_vehiculo  multiples_entregas  precio_base  precio_pedido  anio  mes  dia dia_semana  es_fin_semana
0  0x5b5c  BANGRES15DEL01               24                      4.5   2022-11-28    16:20:00             16:25:00                 Cloudy              Jam        Meal       No  Metropolitian                  26            26.446929             80.377013                  26.486929                   80.417013                   2    

In [ ]:
# ===================================================================
# MEDIDAS DAX → PANDAS
# Cada función recibe un DataFrame; si se le pasa un subconjunto,
# actúa como el "contexto de filtro" de DAX.
# ===================================================================

TRAFICO_PESADO = ['High', 'Jam']

def divide(numerador, denominador, alternativo=0):
    """DIVIDE(num, den, alternativo) de DAX: evita división por cero."""
    return numerador / denominador if denominador else alternativo

# total_pedidos = COUNTROWS(dataset_limpio)
def total_pedidos(d):
    return len(d)

# promedio_tiempo = AVERAGE(dataset_limpio[tiempo_entrega_min])
def promedio_tiempo(d):
    return d['tiempo_entrega_min'].mean()          # ignora vacíos, igual que AVERAGE

# porcentaje_trafico:
#   VAR PedidosTraficoPesado = CALCULATE([Total_Pedidos], densidad_trafico IN {"High","Jam"})
#   RETURN DIVIDE(PedidosTraficoPesado, [Total_Pedidos], 0)
def porcentaje_trafico(d):
    pedidos_trafico_pesado = total_pedidos(d[d['densidad_trafico'].isin(TRAFICO_PESADO)])
    return divide(pedidos_trafico_pesado, total_pedidos(d), 0)

# pedidos_festival = CALCULATE([Total_Pedidos], festival = "Yes")
def pedidos_festival(d):
    return total_pedidos(d[d['festival'] == 'Yes'])

# calificacion_media = AVERAGE(dataset_limpio[calificacion_repartidor])
def calificacion_media(d):
    return d['calificacion_repartidor'].mean()

# ---- KPIs del informe (contexto de filtro: todo el dataset) ----
fecha_col = next((c for c in ['fecha', 'fecha_pedido', 'order_date', 'date']
                  if c in dataset_limpio.columns), None)
if fecha_col:
    f = pd.to_datetime(dataset_limpio[fecha_col], errors='coerce')
    periodo = f"{f.min().strftime('%d/%m/%Y')} - {f.max().strftime('%d/%m/%Y')}"
else:
    periodo = "No disponible (el dataset no tiene columna de fecha)"

kpis = {
    'total_pedidos':          total_pedidos(dataset_limpio),
    'promedio_tiempo':        promedio_tiempo(dataset_limpio),
    'porcentaje_trafico':     porcentaje_trafico(dataset_limpio),          # fracción 0-1
    'porcentaje_trafico_pct': porcentaje_trafico(dataset_limpio) * 100,    # para mostrar
    'pedidos_festival':       pedidos_festival(dataset_limpio),
    'calificacion_media':     calificacion_media(dataset_limpio),
    'periodo':                periodo,
}

print("KPIs (MEDIDAS DAX)")
print("=" * 55)
print(f"total_pedidos:        {kpis['total_pedidos']:>10,}")
print(f"promedio_tiempo:      {kpis['promedio_tiempo']:>10.2f} min")
print(f"porcentaje_trafico:   {kpis['porcentaje_trafico_pct']:>9.1f}%")
print(f"pedidos_festival:     {kpis['pedidos_festival']:>10,}")
print(f"calificacion_media:   {kpis['calificacion_media']:>10.2f}")
print(f"Periodo:              {kpis['periodo']}")
print("=" * 55)

KPIs (MEDIDAS DAX)
total_pedidos:            90,000
promedio_tiempo:           32.04 min
porcentaje_trafico:        40.5%
pedidos_festival:          1,730
calificacion_media:         4.40
Periodo:              01/01/2022 - 10/03/2024


In [ ]:
# ANÁLISIS POR DENSIDAD DE TRÁFICO (las medidas se evalúan en cada segmento)
orden = [x for x in ['Low', 'Medium', 'High', 'Jam'] if x in dataset_limpio['densidad_trafico'].unique()]
orden += [x for x in sorted(dataset_limpio['densidad_trafico'].unique()) if x not in orden]

df_trafico = pd.DataFrame([{
    'densidad_trafico':   g,
    'pedidos':            total_pedidos(dataset_limpio[dataset_limpio['densidad_trafico'] == g]),
    'tiempo_promedio':    promedio_tiempo(dataset_limpio[dataset_limpio['densidad_trafico'] == g]),
    'calificacion_media': calificacion_media(dataset_limpio[dataset_limpio['densidad_trafico'] == g]),
} for g in orden])
df_trafico['pct_pedidos'] = df_trafico['pedidos'] / kpis['total_pedidos'] * 100

print("ANÁLISIS POR DENSIDAD DE TRÁFICO:")
print(df_trafico.round(2).to_string(index=False))

# ANÁLISIS POR FESTIVAL
df_festival = pd.DataFrame([{
    'festival':           g,
    'pedidos':            total_pedidos(dataset_limpio[dataset_limpio['festival'] == g]),
    'tiempo_promedio':    promedio_tiempo(dataset_limpio[dataset_limpio['festival'] == g]),
    'calificacion_media': calificacion_media(dataset_limpio[dataset_limpio['festival'] == g]),
} for g in sorted(dataset_limpio['festival'].unique())])

print("\nANÁLISIS POR FESTIVAL:")
print(df_festival.round(2).to_string(index=False))

# COMPARATIVO TRÁFICO PESADO vs NO PESADO
pesado    = dataset_limpio[dataset_limpio['densidad_trafico'].isin(TRAFICO_PESADO)]
no_pesado = dataset_limpio[~dataset_limpio['densidad_trafico'].isin(TRAFICO_PESADO)]
t_pesado, t_no_pesado = promedio_tiempo(pesado), promedio_tiempo(no_pesado)
c_pesado, c_no_pesado = calificacion_media(pesado), calificacion_media(no_pesado)

print("\nTRÁFICO PESADO (High/Jam) vs RESTO:")
print(f"  Tiempo promedio:    {t_pesado:.2f} vs {t_no_pesado:.2f} min")
print(f"  Calificación media: {c_pesado:.2f} vs {c_no_pesado:.2f}")

ANÁLISIS POR DENSIDAD DE TRÁFICO:
densidad_trafico  pedidos  tiempo_promedio  calificacion_media  pct_pedidos
             Low    30655            32.08                4.40        34.06
          Medium    21743            31.94                4.40        24.16
            High     8715            32.16                4.39         9.68
             Jam    27729            32.06                4.40        30.81
     Desconocido     1158            31.68                4.41         1.29

ANÁLISIS POR FESTIVAL:
festival  pedidos  tiempo_promedio  calificacion_media
      No    88270            32.04                 4.4
     Yes     1730            31.95                 4.4

TRÁFICO PESADO (High/Jam) vs RESTO:
  Tiempo promedio:    32.08 vs 32.01 min
  Calificación media: 4.40 vs 4.40


In [ ]:
# ===================================================================
# GRÁFICOS (se guardan en evidencias/reporte)
# ===================================================================
dir_graf = BASE / 'evidencias' / 'reporte'
colores_trafico = {'Low': '#2ecc71', 'Medium': '#f1c40f', 'High': '#e67e22', 'Jam': '#e74c3c'}

def etiquetar(ax, barras, fmt):
    for b in barras:
        ax.text(b.get_x() + b.get_width()/2., b.get_height(), fmt.format(b.get_height()),
                ha='center', va='bottom', fontsize=11, fontweight='bold')

# GRÁFICO 1: Pedidos por densidad de tráfico  (total_pedidos)
fig, ax = plt.subplots(figsize=(10, 6))
cols = [colores_trafico.get(g, '#2c7bb6') for g in df_trafico['densidad_trafico']]
barras = ax.bar(df_trafico['densidad_trafico'], df_trafico['pedidos'],
                color=cols, edgecolor='#2c3e50', linewidth=1.5)
ax.set_title('Total de Pedidos por Densidad de Tráfico', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Densidad de tráfico'); ax.set_ylabel('Pedidos')
ax.grid(True, alpha=0.3, axis='y', linestyle='--')
etiquetar(ax, barras, '{:,.0f}')
plt.tight_layout()
ruta_g1 = dir_graf / 'grafico_pedidos_trafico.png'
plt.savefig(ruta_g1, dpi=150, bbox_inches='tight', facecolor='white'); plt.close()
print(f"------ Gráfico 1: {ruta_g1.name}")

# GRÁFICO 2: Tiempo promedio por densidad de tráfico  (promedio_tiempo)
fig, ax = plt.subplots(figsize=(10, 6))
barras = ax.bar(df_trafico['densidad_trafico'], df_trafico['tiempo_promedio'],
                color=cols, edgecolor='#2c3e50', linewidth=1.5)
ax.axhline(kpis['promedio_tiempo'], color='#1a5490', linestyle='--', linewidth=2,
           label=f"Promedio general: {kpis['promedio_tiempo']:.1f} min")
ax.set_title('Tiempo Promedio de Entrega por Densidad de Tráfico', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Densidad de tráfico'); ax.set_ylabel('Minutos')
ax.grid(True, alpha=0.3, axis='y', linestyle='--'); ax.legend()
etiquetar(ax, barras, '{:.1f}')
plt.tight_layout()
ruta_g2 = dir_graf / 'grafico_tiempo_trafico.png'
plt.savefig(ruta_g2, dpi=150, bbox_inches='tight', facecolor='white'); plt.close()
print(f"------ Gráfico 2: {ruta_g2.name}")

# GRÁFICO 3: Festival vs No festival  (pedidos_festival y promedio_tiempo)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
col_fest = ['#e74c3c' if g == 'Yes' else '#3498db' for g in df_festival['festival']]
b1 = axes[0].bar(df_festival['festival'], df_festival['pedidos'], color=col_fest, edgecolor='#2c3e50', linewidth=1.5)
axes[0].set_title('Pedidos: Festival vs No Festival', fontweight='bold'); axes[0].set_ylabel('Pedidos')
etiquetar(axes[0], b1, '{:,.0f}')
b2 = axes[1].bar(df_festival['festival'], df_festival['tiempo_promedio'], color=col_fest, edgecolor='#2c3e50', linewidth=1.5)
axes[1].set_title('Tiempo promedio de entrega (min)', fontweight='bold'); axes[1].set_ylabel('Minutos')
etiquetar(axes[1], b2, '{:.1f}')
for a in axes: a.grid(True, alpha=0.3, axis='y', linestyle='--')
plt.tight_layout()
ruta_g3 = dir_graf / 'grafico_festival.png'
plt.savefig(ruta_g3, dpi=150, bbox_inches='tight', facecolor='white'); plt.close()
print(f"------ Gráfico 3: {ruta_g3.name}")

# GRÁFICO 4: Distribución de calificación del repartidor  (calificacion_media)
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(dataset_limpio['calificacion_repartidor'].dropna(), bins=15,
        color='#2c7bb6', edgecolor='#1a5490', alpha=0.85)
ax.axvline(kpis['calificacion_media'], color='#e74c3c', linestyle='--', linewidth=2.5,
           label=f"Calificación media: {kpis['calificacion_media']:.2f}")
ax.set_title('Distribución de la Calificación del Repartidor', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Calificación'); ax.set_ylabel('Pedidos')
ax.grid(True, alpha=0.3, axis='y', linestyle='--'); ax.legend()
plt.tight_layout()
ruta_g4 = dir_graf / 'grafico_calificacion.png'
plt.savefig(ruta_g4, dpi=150, bbox_inches='tight', facecolor='white'); plt.close()
print(f"------ Gráfico 4: {ruta_g4.name}")

print("\n¡Los 4 gráficos se generaron correctamente!")

------ Gráfico 1: grafico_pedidos_trafico.png
------ Gráfico 2: grafico_tiempo_trafico.png
------ Gráfico 3: grafico_festival.png
------ Gráfico 4: grafico_calificacion.png

¡Los 4 gráficos se generaron correctamente!


In [ ]:
# VERIFICAR Y DESCARGAR LOS GRÁFICOS
graficos = {'Pedidos x tráfico': ruta_g1, 'Tiempo x tráfico': ruta_g2,
            'Festival': ruta_g3, 'Calificación': ruta_g4}

print("VERIFICANDO GRÁFICOS:\n")
for nombre, ruta in graficos.items():
    if ruta.exists():
        print(f"OK {nombre}: {ruta.name} ({ruta.stat().st_size / 1024:.1f} KB)")
    else:
        print(f"FAILED {nombre}: NO ENCONTRADO")

try:
    from google.colab import files
    print("\nDescargando gráficos...")
    for ruta in graficos.values():
        files.download(str(ruta))
except ImportError:
    files = None   # fuera de Colab no se descarga

VERIFICANDO GRÁFICOS:

OK Pedidos x tráfico: grafico_pedidos_trafico.png (55.0 KB)
OK Tiempo x tráfico: grafico_tiempo_trafico.png (58.4 KB)
OK Festival: grafico_festival.png (49.8 KB)
OK Calificación: grafico_calificacion.png (50.3 KB)

Descargando gráficos...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# TEMPLATE HTML CON JINJA2
template_informe = """
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Informe Gerencial - {{ empresa }}</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 20px; color: #333; line-height: 1.35; }
        h1 { color: #1a5490; border-bottom: 3px solid #1a5490; padding-bottom: 6px; }
        h2 { color: #2c7bb6; margin-top: 16px; border-bottom: 1px solid #ddd; padding-bottom: 3px; }
        h3 { color: #444; margin-top: 10px; }
        .portada { text-align: center; margin-bottom: 15px; padding: 12px; background: #f0f7ff; border-radius: 8px; }
        .portada h1 { border: none; font-size: 28px; margin-bottom: 3px; }
        .portada h2 { border: none; color: #666; margin-top: 3px; }
        .kpi-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; margin: 10px 0; }
        .kpi-card { background: #f0f7ff; padding: 8px; border-left: 4px solid #2c7bb6; border-radius: 4px; }
        .kpi-valor { font-size: 24px; font-weight: bold; color: #1a5490; }
        .kpi-label { font-size: 12px; color: #666; text-transform: uppercase; margin-bottom: 3px; }
        .hallazgo { background: #f9f9f9; padding: 10px; border-left: 4px solid #ff9800; margin: 10px 0; border-radius: 4px; }
        .hallazgo h3 { margin-top: 0; color: #e65100; }
        .limitacion { background: #fff3cd; padding: 5px 8px; border-left: 4px solid #ffc107; margin: 5px 0; border-radius: 4px; }
        .limitacion-titulo { font-weight: bold; color: #856404; margin-right: 5px; }
        table { border-collapse: collapse; width: 100%; margin: 8px 0; }
        th, td { border: 1px solid #ddd; padding: 5px; text-align: left; }
        th { background: #2c7bb6; color: white; }
        tr:nth-child(even) { background: #f9f9f9; }
        code { font-family: Consolas, monospace; font-size: 11px; white-space: pre-wrap; }
        .pie { font-size: 11px; color: #999; margin-top: 20px; border-top: 1px solid #ddd; padding-top: 6px; text-align: center; }
        img { max-width: 100%; margin: 6px 0; border: 1px solid #eee; border-radius: 4px; }
        .metodologia { background: #e8f5e9; padding: 8px; border-radius: 4px; margin: 8px 0; }
    </style>
</head>
<body>

<div class="portada">
    <h1>INFORME GERENCIAL DE OPERACIONES DE ENTREGA</h1>
    <h2>{{ empresa }}</h2>
    <p><strong>Periodo:</strong> {{ periodo }}</p>
    <p><strong>Fecha de generación:</strong> {{ fecha_generacion }}</p>
    <p><strong>Versión:</strong> {{ version }}</p>
    <p><strong>Equipo:</strong> {{ equipo }}</p>
</div>

<h2>1. Resumen Ejecutivo</h2>
<p>El presente informe analiza la operación de entregas de {{ empresa }} a partir de {{ kpis.total_pedidos }} pedidos
registrados en el dataset protegido (Guía 4). Su propósito es apoyar la decisión de cómo gestionar los tiempos de entrega
en condiciones de tráfico pesado y en días de festival. Los indicadores corresponden a las medidas DAX del modelo en Power BI.</p>

<h3>Indicadores Principales (medidas DAX)</h3>
<div class="kpi-grid">
    <div class="kpi-card">
        <div class="kpi-label">Total de Pedidos</div>
        <div class="kpi-valor">{{ "{:,}".format(kpis.total_pedidos) }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Tiempo Promedio (min)</div>
        <div class="kpi-valor">{{ "%.1f"|format(kpis.promedio_tiempo) }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">% Pedidos en Tráfico Pesado</div>
        <div class="kpi-valor">{{ "%.1f"|format(kpis.porcentaje_trafico_pct) }}%</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Pedidos en Festival</div>
        <div class="kpi-valor">{{ "{:,}".format(kpis.pedidos_festival) }}</div>
    </div>
    <div class="kpi-card">
        <div class="kpi-label">Calificación Media del Repartidor</div>
        <div class="kpi-valor">{{ "%.2f"|format(kpis.calificacion_media) }}</div>
    </div>
</div>

<h2>2. Contexto y Metodología</h2>
<div class="metodologia">
    <p><strong>Fuente utilizada:</strong> dataset_protegido.csv (Guía 4 - Gobernanza), tabla <em>dataset_limpio</em> del modelo.</p>
    <p><strong>Proceso:</strong> Integración → Limpieza → Procesamiento → Gobernanza → Medidas DAX → Reportería automatizada.</p>
    <p><strong>Filtros aplicados:</strong> Periodo: {{ periodo }}; todas las densidades de tráfico; festival y no festival.</p>
    <p><strong>Definición de tráfico pesado:</strong> densidad_trafico igual a "High" o "Jam".</p>
</div>

<h3>Medidas DAX utilizadas</h3>
<table>
    <thead><tr><th>Medida</th><th>Expresión DAX</th><th>Valor</th></tr></thead>
    <tbody>
        {% for m in medidas_dax %}
        <tr><td>{{ m.nombre }}</td><td><code>{{ m.dax }}</code></td><td>{{ m.valor }}</td></tr>
        {% endfor %}
    </tbody>
</table>

<h2>3. Resultados Generales</h2>

<h3>3.1 Pedidos por Densidad de Tráfico</h3>
<img src="{{ img_pedidos_trafico }}" alt="Pedidos por densidad de tráfico">
<p><strong>Lectura:</strong> La densidad de tráfico con más pedidos es {{ trafico_top }} ({{ "{:,}".format(pedidos_trafico_top) }} pedidos,
{{ "%.1f"|format(pct_trafico_top) }}% del total). En conjunto, el tráfico pesado (High/Jam) representa el {{ "%.1f"|format(kpis.porcentaje_trafico_pct) }}% de los pedidos.</p>

<h3>3.2 Tiempo de Entrega por Densidad de Tráfico</h3>
<img src="{{ img_tiempo_trafico }}" alt="Tiempo promedio por densidad de tráfico">
<p><strong>Lectura:</strong> El tiempo promedio general es de {{ "%.1f"|format(kpis.promedio_tiempo) }} minutos.
En tráfico pesado el promedio es de {{ t_pesado_txt }} minutos, frente a {{ t_no_pesado_txt }} minutos en el resto de los pedidos.</p>

<h3>3.3 Festival vs No Festival</h3>
<img src="{{ img_festival }}" alt="Festival vs no festival">
<p><strong>Lectura:</strong> {{ "{:,}".format(kpis.pedidos_festival) }} pedidos ({{ "%.1f"|format(pct_festival) }}% del total) se realizaron en días de festival.
El tiempo promedio en festival es de {{ t_festival_txt }} minutos, frente a {{ t_no_festival_txt }} minutos fuera de festival.</p>

<h2>4. Calidad del Servicio</h2>
<img src="{{ img_calificacion }}" alt="Distribución de calificación">
<p><strong>Patrón observado:</strong> La calificación media del repartidor es {{ "%.2f"|format(kpis.calificacion_media) }}.
En tráfico pesado es {{ c_pesado_txt }}, frente a {{ c_no_pesado_txt }} en el resto de los pedidos.</p>
<p><strong>Lectura responsable:</strong> Estas comparaciones describen asociaciones; no demuestran que el tráfico o el festival sean la causa de las diferencias.</p>

<h2>5. Hallazgos Principales</h2>

{% for h in hallazgos %}
<div class="hallazgo">
    <h3>{{ loop.index }}. {{ h.titulo }}</h3>
    <p><strong>Evidencia:</strong> {{ h.evidencia }}</p>
    <p><strong>Interpretación:</strong> {{ h.interpretacion }}</p>
    <p><strong>Impacto:</strong> {{ h.impacto }}</p>
    <p><strong>Decisión sugerida:</strong> {{ h.decision }}</p>
    <p><em>Limitación: {{ h.limitacion }}</em></p>
</div>
{% endfor %}

<h2>6. Recomendaciones</h2>
<table>
    <thead>
        <tr><th>#</th><th>Acción</th><th>Prioridad</th><th>Responsable</th><th>Plazo</th></tr>
    </thead>
    <tbody>
        {% for r in recomendaciones %}
        <tr>
            <td>{{ loop.index }}</td>
            <td>{{ r.accion }}</td>
            <td>{{ r.prioridad }}</td>
            <td>{{ r.responsable }}</td>
            <td>{{ r.plazo }}</td>
        </tr>
        {% endfor %}
    </tbody>
</table>

<h2>7. Limitaciones del Análisis</h2>
{% for l in limitaciones %}
<div class="limitacion"><span class="limitacion-titulo">[!]</span> {{ l }}</div>
{% endfor %}

<h2>8. Conclusiones</h2>
<p>El análisis de {{ kpis.total_pedidos }} pedidos muestra un tiempo de entrega promedio de {{ "%.1f"|format(kpis.promedio_tiempo) }} minutos
y una calificación media de {{ "%.2f"|format(kpis.calificacion_media) }}. El {{ "%.1f"|format(kpis.porcentaje_trafico_pct) }}% de los pedidos
se realizó con tráfico pesado y {{ "{:,}".format(kpis.pedidos_festival) }} durante festivales, por lo que ambos factores deben
considerarse en la planificación operativa.</p>

<div class="pie">
    <p>Informe generado automáticamente por el pipeline de reportería - {{ empresa }}</p>
    <p>Proyecto integrador COIDS1604 - Guía Práctica N.º 7 (versión DAX)</p>
</div>

</body>
</html>
"""

print("Template Jinja2 definido")
print(f"Longitud: {len(template_informe):,} caracteres")

Template Jinja2 definido
Longitud: 8,280 caracteres


In [ ]:
# FUNCIONES AUXILIARES
def imagen_a_base64(ruta):
    """Convierte una imagen a base64 para incrustarla directamente en HTML."""
    with open(ruta, 'rb') as f:
        return f"data:image/png;base64,{base64.b64encode(f.read()).decode()}"

def num(x, dec=1):
    """Formatea un número; si es NaN (segmento vacío) devuelve 'N/D'."""
    return 'N/D' if pd.isna(x) else f"{x:.{dec}f}"

def comparar(a, b, mayor='mayor', menor='menor', igual='igual'):
    """Texto comparativo seguro cuando alguno de los valores es NaN."""
    if pd.isna(a) or pd.isna(b): return 'no comparable'
    return mayor if a > b else (menor if a < b else igual)

# VALORES DERIVADOS PARA EL TEXTO DEL INFORME
fila_top = df_trafico.sort_values('pedidos', ascending=False).iloc[0]
festival  = dataset_limpio[dataset_limpio['festival'] == 'Yes']
no_festival = dataset_limpio[dataset_limpio['festival'] != 'Yes']
t_fest, t_nofest = promedio_tiempo(festival), promedio_tiempo(no_festival)
pct_festival = divide(kpis['pedidos_festival'], kpis['total_pedidos'], 0) * 100

# MEDIDAS DAX (texto + valor) PARA LA TABLA DEL INFORME
medidas_dax = [
    {'nombre': 'total_pedidos',      'dax': 'COUNTROWS(dataset_limpio)',
     'valor': f"{kpis['total_pedidos']:,}"},
    {'nombre': 'promedio_tiempo',    'dax': 'AVERAGE(dataset_limpio[tiempo_entrega_min])',
     'valor': f"{kpis['promedio_tiempo']:.2f} min"},
    {'nombre': 'porcentaje_trafico', 'dax': 'VAR PedidosTraficoPesado = CALCULATE([Total_Pedidos], dataset_limpio[densidad_trafico] IN {"High", "Jam"}) RETURN DIVIDE(PedidosTraficoPesado, [Total_Pedidos], 0)',
     'valor': f"{kpis['porcentaje_trafico_pct']:.1f}%"},
    {'nombre': 'pedidos_festival',   'dax': 'CALCULATE([Total_Pedidos], dataset_limpio[festival] = "Yes")',
     'valor': f"{kpis['pedidos_festival']:,}"},
    {'nombre': 'calificacion_media', 'dax': 'AVERAGE(dataset_limpio[calificacion_repartidor])',
     'valor': f"{kpis['calificacion_media']:.2f}"},
]

# PREPARAR DATOS PARA EL TEMPLATE
datos_template = {
    'empresa': 'DataRetail AI',
    'periodo': kpis['periodo'],
    'fecha_generacion': datetime.now().strftime('%d/%m/%Y %H:%M'),
    'version': '1.0',
    'equipo': 'Equipo DataRetail AI',
    'kpis': SimpleNamespace(**kpis),
    'medidas_dax': medidas_dax,

    # Imágenes en base64
    'img_pedidos_trafico': imagen_a_base64(ruta_g1),
    'img_tiempo_trafico':  imagen_a_base64(ruta_g2),
    'img_festival':        imagen_a_base64(ruta_g3),
    'img_calificacion':    imagen_a_base64(ruta_g4),

    # Datos para texto
    'trafico_top': fila_top['densidad_trafico'],
    'pedidos_trafico_top': int(fila_top['pedidos']),
    'pct_trafico_top': fila_top['pct_pedidos'],
    't_pesado_txt': num(t_pesado), 't_no_pesado_txt': num(t_no_pesado),
    'c_pesado_txt': num(c_pesado, 2), 'c_no_pesado_txt': num(c_no_pesado, 2),
    't_festival_txt': num(t_fest), 't_no_festival_txt': num(t_nofest),
    'pct_festival': pct_festival,

    # Hallazgos (todo calculado a partir de las medidas)
    'hallazgos': [
        {
            'titulo': f"El tráfico pesado concentra el {kpis['porcentaje_trafico_pct']:.1f}% de los pedidos",
            'evidencia': f"{int(round(kpis['porcentaje_trafico'] * kpis['total_pedidos'])):,} de {kpis['total_pedidos']:,} pedidos se entregaron con densidad High o Jam. "
                         f"El tiempo promedio fue {num(t_pesado)} min en tráfico pesado vs {num(t_no_pesado)} min en el resto.",
            'interpretacion': f"El tiempo de entrega en tráfico pesado es {comparar(t_pesado, t_no_pesado)} que en condiciones normales.",
            'impacto': 'Una parte relevante de la operación depende de condiciones de tráfico que la empresa no controla.',
            'decision': 'Ajustar la promesa de tiempo de entrega según la densidad de tráfico.',
            'limitacion': 'No se controla por distancia ni por zona de entrega.'
        },
        {
            'titulo': f"{kpis['pedidos_festival']:,} pedidos se realizaron en festival ({pct_festival:.1f}% del total)",
            'evidencia': f"Tiempo promedio en festival: {num(t_fest)} min; fuera de festival: {num(t_nofest)} min.",
            'interpretacion': f"Los pedidos en festival tardan {comparar(t_fest, t_nofest)} en promedio que los demás.",
            'impacto': 'Los festivales pueden requerir más repartidores o ventanas de entrega más amplias.',
            'decision': 'Planificar capacidad adicional en fechas de festival.',
            'limitacion': 'El indicador no distingue entre festivales de distinto tamaño.'
        },
        {
            'titulo': f"La calificación media del repartidor es {kpis['calificacion_media']:.2f}",
            'evidencia': f"En tráfico pesado la calificación media es {num(c_pesado, 2)}; en el resto, {num(c_no_pesado, 2)}.",
            'interpretacion': f"La calificación en tráfico pesado es {comparar(c_pesado, c_no_pesado)} que en el resto de los pedidos.",
            'impacto': 'La calificación es un indicador de la experiencia del cliente y de la reputación del servicio.',
            'decision': 'Dar seguimiento a la calificación junto con el tiempo de entrega.',
            'limitacion': 'La calificación es una percepción subjetiva y puede variar entre clientes.'
        }
    ],

    # Recomendaciones
    'recomendaciones': [
        {'accion': 'Ajustar el tiempo estimado de entrega según densidad de tráfico (High/Jam)', 'prioridad': 'Alta', 'responsable': 'Gerente de Operaciones', 'plazo': 'Mensual'},
        {'accion': 'Reforzar la capacidad de repartidores en días de festival', 'prioridad': 'Alta', 'responsable': 'Jefe de Logística', 'plazo': 'Antes del próximo festival'},
        {'accion': 'Monitorear la calificación media junto con el tiempo promedio de entrega', 'prioridad': 'Media', 'responsable': 'Analista de Datos', 'plazo': 'Semanal'},
        {'accion': 'Validar los patrones con más periodos y variables (distancia, zona)', 'prioridad': 'Media', 'responsable': 'Analista de Datos', 'plazo': 'Próximo periodo'},
    ],

    # Limitaciones
    'limitaciones': [
        f"Volumen de datos: {kpis['total_pedidos']:,} pedidos en el dataset analizado.",
        'Dataset protegido: puede omitir o anonimizar columnas por gobernanza (Guía 4).',
        'Tráfico pesado se define como densidad "High" o "Jam"; otra definición cambiaría el porcentaje.',
        'No se controla por distancia, clima ni zona de entrega.',
        'Correlación no implica causalidad: los patrones son asociaciones.'
    ]
}

# RENDERIZAR TEMPLATE
template = Template(template_informe)
html_renderizado = template.render(**datos_template)

ruta_html = BASE / 'exportaciones' / 'informe_entregas.html'
ruta_html.write_text(html_renderizado, encoding='utf-8')
print(f"HTML generado: {ruta_html.name}")

# CONVERTIR A PDF
from weasyprint import HTML
ruta_pdf = BASE / 'exportaciones' / 'informe_entregas_v1.0.pdf'
HTML(string=html_renderizado).write_pdf(str(ruta_pdf))
print(f"PDF generado: {ruta_pdf.name}  ({ruta_pdf.stat().st_size / 1024:.1f} KB)")

try:
    from google.colab import files
    files.download(str(ruta_pdf))
    print("PDF descargado")
except ImportError:
    pass

HTML generado: informe_entregas.html
PDF generado: informe_entregas_v1.0.pdf  (237.4 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PDF descargado


In [ ]:
# GENERAR DOCX CON PYTHON-DOCX
from docx import Document
from docx.shared import Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH

doc = Document()

titulo = doc.add_heading('INFORME GERENCIAL DE OPERACIONES DE ENTREGA', 0)
titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER
subtitulo = doc.add_heading(datos_template['empresa'], level=1)
subtitulo.alignment = WD_ALIGN_PARAGRAPH.CENTER

doc.add_paragraph(f'Periodo: {kpis["periodo"]}')
doc.add_paragraph(f'Fecha de generación: {datetime.now().strftime("%d/%m/%Y %H:%M")}')
doc.add_paragraph('Versión: 1.0')

# Resumen ejecutivo
doc.add_heading('1. Resumen Ejecutivo', level=2)
doc.add_paragraph(f'El presente informe analiza {kpis["total_pedidos"]:,} pedidos del dataset protegido, '
                  f'con los indicadores definidos como medidas DAX en el modelo.')

# KPIs (medidas DAX)
doc.add_heading('Indicadores Principales (medidas DAX)', level=3)
datos_kpis = [
    ('Total de Pedidos',                   f'{kpis["total_pedidos"]:,}'),
    ('Tiempo Promedio (min)',              f'{kpis["promedio_tiempo"]:.1f}'),
    ('% Pedidos en Tráfico Pesado',        f'{kpis["porcentaje_trafico_pct"]:.1f}%'),
    ('Pedidos en Festival',                f'{kpis["pedidos_festival"]:,}'),
    ('Calificación Media del Repartidor',  f'{kpis["calificacion_media"]:.2f}'),
]
tabla_kpis = doc.add_table(rows=len(datos_kpis), cols=2)
tabla_kpis.style = 'Light Grid Accent 1'
for i, (label, valor) in enumerate(datos_kpis):
    tabla_kpis.rows[i].cells[0].text = label
    tabla_kpis.rows[i].cells[1].text = valor

# Medidas DAX
doc.add_heading('Medidas DAX utilizadas', level=3)
tabla_dax = doc.add_table(rows=1, cols=2)
tabla_dax.style = 'Light Grid Accent 1'
tabla_dax.rows[0].cells[0].text = 'Medida'
tabla_dax.rows[0].cells[1].text = 'Expresión DAX'
for m in medidas_dax:
    row = tabla_dax.add_row()
    row.cells[0].text = m['nombre']
    row.cells[1].text = m['dax']

# Resultados con gráficos
doc.add_heading('2. Resultados Generales', level=2)
doc.add_heading('2.1 Pedidos por Densidad de Tráfico', level=3)
doc.add_picture(str(ruta_g1), width=Inches(6))
doc.add_heading('2.2 Tiempo de Entrega por Densidad de Tráfico', level=3)
doc.add_picture(str(ruta_g2), width=Inches(6))
doc.add_heading('2.3 Festival vs No Festival', level=3)
doc.add_picture(str(ruta_g3), width=Inches(6))
doc.add_heading('2.4 Calificación del Repartidor', level=3)
doc.add_picture(str(ruta_g4), width=Inches(6))

# Hallazgos
doc.add_heading('3. Hallazgos Principales', level=2)
for i, h in enumerate(datos_template['hallazgos'], 1):
    doc.add_heading(f'{i}. {h["titulo"]}', level=3)
    doc.add_paragraph(f'Evidencia: {h["evidencia"]}')
    doc.add_paragraph(f'Interpretación: {h["interpretacion"]}')
    doc.add_paragraph(f'Impacto: {h["impacto"]}')
    doc.add_paragraph(f'Decisión: {h["decision"]}')

# Recomendaciones
doc.add_heading('4. Recomendaciones', level=2)
tabla_rec = doc.add_table(rows=1, cols=5)
tabla_rec.style = 'Light Grid Accent 1'
for i, h in enumerate(['#', 'Acción', 'Prioridad', 'Responsable', 'Plazo']):
    tabla_rec.rows[0].cells[i].text = h
for i, r in enumerate(datos_template['recomendaciones'], 1):
    row = tabla_rec.add_row()
    row.cells[0].text = str(i)
    row.cells[1].text = r['accion']
    row.cells[2].text = r['prioridad']
    row.cells[3].text = r['responsable']
    row.cells[4].text = r['plazo']

# Limitaciones
doc.add_heading('5. Limitaciones', level=2)
for l in datos_template['limitaciones']:
    doc.add_paragraph(l, style='List Bullet')

ruta_docx = BASE / 'exportaciones' / 'informe_entregas_v1.0.docx'
doc.save(str(ruta_docx))
print(f"DOCX generado: {ruta_docx.name}  ({ruta_docx.stat().st_size / 1024:.1f} KB)")

try:
    from google.colab import files
    files.download(str(ruta_docx))
    print("DOCX descargado")
except ImportError:
    pass

DOCX generado: informe_entregas_v1.0.docx  (209.1 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

DOCX descargado


In [ ]:
# METADATA DE TRAZABILIDAD
metadata = {
    'version': '1.0',
    'fecha_generacion': datetime.now().isoformat(),
    'periodo_analizado': kpis['periodo'],
    'fuentes_datos': ['dataset_protegido.csv (Guía 4 - gobernanza)'],
    'filtros_aplicados': {
        'periodo': kpis['periodo'],
        'densidad_trafico': 'todas',
        'festival': 'todos'
    },
    'medidas_dax': {m['nombre']: m['dax'] for m in medidas_dax},
    'kpis': {
        'total_pedidos':      int(kpis['total_pedidos']),
        'promedio_tiempo':    float(kpis['promedio_tiempo']),
        'porcentaje_trafico': float(kpis['porcentaje_trafico']),
        'pedidos_festival':   int(kpis['pedidos_festival']),
        'calificacion_media': float(kpis['calificacion_media'])
    },
    'archivos_generados': [ruta_pdf.name, ruta_docx.name, ruta_html.name],
    'generado_por': 'pipeline_reporteria_guia07_dax.py'
}

ruta_metadata = BASE / 'exportaciones' / 'metadata_informe.json'
ruta_metadata.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

print("Metadata de trazabilidad guardada:")
print(json.dumps(metadata, indent=2, ensure_ascii=False))

Metadata de trazabilidad guardada:
{
  "version": "1.0",
  "fecha_generacion": "2026-11-19T04:23:02.943187",
  "periodo_analizado": "01/01/2022 - 10/03/2024",
  "fuentes_datos": [
    "dataset_protegido.csv (Guía 4 - gobernanza)"
  ],
  "filtros_aplicados": {
    "periodo": "01/01/2022 - 10/03/2024",
    "densidad_trafico": "todas",
    "festival": "todos"
  },
  "medidas_dax": {
    "total_pedidos": "COUNTROWS(dataset_limpio)",
    "promedio_tiempo": "AVERAGE(dataset_limpio[tiempo_entrega_min])",
    "porcentaje_trafico": "VAR PedidosTraficoPesado = CALCULATE([Total_Pedidos], dataset_limpio[densidad_trafico] IN {\"High\", \"Jam\"}) RETURN DIVIDE(PedidosTraficoPesado, [Total_Pedidos], 0)",
    "pedidos_festival": "CALCULATE([Total_Pedidos], dataset_limpio[festival] = \"Yes\")",
    "calificacion_media": "AVERAGE(dataset_limpio[calificacion_repartidor])"
  },
  "kpis": {
    "total_pedidos": 90000,
    "promedio_tiempo": 32.041755555555554,
    "porcentaje_trafico": 0.4049333333333333,
